# Statystyka dla Data Analyst/Engineer — Moduł 13: Regresja logistyczna

Regresja liniowa z Modułu 12 przewiduje zmienną CIĄGŁĄ. Ale bardzo często interesuje nas
wynik BINARNY: czy klient dokona konwersji (tak/nie), czy transakcja jest oszustwem
(tak/nie), czy klient odejdzie (*churn*, tak/nie). **Regresja logistyczna** to
standardowe narzędzie do takich zadań — wykorzystamy naszą kolumnę `konwersja`.

## Spis treści
1. [Dlaczego nie zwykła regresja liniowa?](#sec1)
2. [Funkcja logistyczna i logit](#sec2)
3. [Dopasowanie regresji logistycznej](#sec3)
4. [Interpretacja: iloraz szans](#sec4)
5. [Przewidywanie prawdopodobieństw](#sec5)
6. [Próg decyzyjny i macierz pomyłek](#sec6)
7. [Precyzja i czułość](#sec7)
8. [Podsumowanie i ćwiczenia](#sec8)


Każdy moduł tego kursu zaczyna się od wygenerowania tego samego zestawu danych syntetycznych -- dzięki ustalonemu ziarnu losowości (`np.random.default_rng(42)`) liczby zawsze wyjdą identyczne, więc możesz otworzyć dowolny moduł niezależnie od pozostałych, nawet po restarcie kernela.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400

miasto = rng.choice(
    ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"],
    size=n, p=[0.30, 0.25, 0.15, 0.15, 0.15],
)
region = rng.choice(["Polnoc", "Poludnie", "Wschod", "Zachod"], size=n)
wiek = rng.integers(18, 70, size=n)

# wydatki miesieczne: rozklad prawoskosny (typowy dla danych o wydatkach)
wydatki_miesieczne = rng.lognormal(mean=5.2, sigma=0.5, size=n).round(2)

# satysfakcja skorelowana z wydatkami + efekt regionu (celowo rozny) + szum losowy,
# przyciete do skali 1-10 -- dzieki temu Modul 10 (ANOVA) ma realna roznice miedzy grupami
efekt_regionu = {"Polnoc": 0.0, "Poludnie": -0.5, "Wschod": 0.3, "Zachod": 0.8}
satysfakcja = np.clip(
    wydatki_miesieczne / 45
    + np.array([efekt_regionu[r] for r in region])
    + rng.normal(0, 1.5, size=n),
    1, 10,
).round(1)

# grupa kampanii A/B -- z celowo wprowadzonym efektem w konwersji (grupa B lepsza)
grupa = rng.choice(["A", "B"], size=n)
prawdopodobienstwo_konwersji = np.where(grupa == "A", 0.10, 0.20)
konwersja = rng.binomial(1, prawdopodobienstwo_konwersji)

# wydatki przed/po wprowadzeniu programu lojalnosciowego (dane sparowane)
wydatki_przed = rng.normal(220, 40, size=n).round(2)
efekt_programu = rng.normal(15, 10, size=n)
wydatki_po = (wydatki_przed + efekt_programu).round(2)

klienci = pd.DataFrame({
    "klient_id": np.arange(1, n + 1),
    "miasto": miasto,
    "region": region,
    "wiek": wiek,
    "wydatki_miesieczne": wydatki_miesieczne,
    "satysfakcja": satysfakcja,
    "grupa_kampanii": grupa,
    "konwersja": konwersja,
    "wydatki_przed": wydatki_przed,
    "wydatki_po": wydatki_po,
})
klienci.head()


<a id="sec1"></a>
## 1. Dlaczego nie zwykła regresja liniowa?

`konwersja` przyjmuje tylko wartości `0` albo `1`. Spróbujmy mimo to dopasować do niej
zwykłą regresję liniową (Moduł 12) względem `wydatki_miesieczne` i zobaczmy, co się
stanie dla wysokich wartości predyktora:

In [ ]:
from statsmodels.formula.api import ols

model_liniowy = ols("konwersja ~ wydatki_miesieczne", data=klienci).fit()
print(model_liniowy.params)

przewidywanie_ekstremalne = model_liniowy.predict(pd.DataFrame({"wydatki_miesieczne": [2000.0]}))
print(f"Przewidywane 'prawdopodobienstwo' konwersji przy wydatkach=2000: {przewidywanie_ekstremalne.values[0]:.3f}")


> ⚠️ **Ujemne prawdopodobieństwo nie ma sensu**
>
> Zwykła regresja liniowa nie ma ŻADNEGO wbudowanego ograniczenia — linia prosta swobodnie wychodzi poza przedział `[0, 1]`, dając bezsensowne "prawdopodobieństwa" ujemne albo większe od 1. Potrzebujemy modelu, który z definicji zwraca wartości wyłącznie w przedziale `[0, 1]`.

<a id="sec2"></a>
## 2. Funkcja logistyczna i logit

**Funkcja logistyczna** (*sigmoid*) przekształca dowolną liczbę rzeczywistą na wartość w
przedziale `(0, 1)`:

```
p = 1 / (1 + e^(-z))
```

gdzie `z = b0 + b1*x1 + b2*x2 + ...` (dokładnie ta sama liniowa kombinacja predyktorów co
w regresji liniowej!). Odwrotność tej funkcji to **logit** (log szans):

```
logit(p) = ln(p / (1 - p)) = z
```

Innymi słowy: regresja logistyczna modeluje LINIOWO nie samo prawdopodobieństwo `p`
(które musi mieścić się w `[0, 1]`), tylko *logit* tego prawdopodobieństwa (który może
przyjąć dowolną wartość rzeczywistą).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

z = np.linspace(-8, 8, 200)
p = 1 / (1 + np.exp(-z))

plt.figure(figsize=(7, 5))
plt.plot(z, p)
plt.axhline(0.5, color="gray", linestyle="--", alpha=0.5)
plt.axvline(0, color="gray", linestyle="--", alpha=0.5)
plt.xlabel("z (kombinacja liniowa predyktorow)")
plt.ylabel("p (prawdopodobienstwo)")
plt.title("Funkcja logistyczna (sigmoid)")
plt.show()


<a id="sec3"></a>
## 3. Dopasowanie regresji logistycznej

Sprawdźmy, czy `grupa_kampanii` (A/B) wpływa na `konwersja` — dokładnie to pytanie, które
badaliśmy testem chi-kwadrat w Module 9 i przedziałem ufności dla proporcji w Module 6.

In [ ]:
from statsmodels.formula.api import logit

model = logit("konwersja ~ C(grupa_kampanii)", data=klienci).fit(disp=0)
print(model.summary())


> 📊 **Regresja logistyczna z jedną zmienną kategoryczną = test proporcji**
>
> Gdy jedynym predyktorem jest zmienna kategoryczna dwupoziomowa, regresja logistyczna i test chi-kwadrat/proporcji z Modułów 6 i 9 odpowiadają na to samo pytanie z bardzo podobnym wynikiem. Prawdziwa siła regresji logistycznej ujawnia się, gdy chcemy uwzględnić WIELE predyktorów naraz (sekcja 4 dalej) — coś, czego prosty test chi-kwadrat nie potrafi zrobić.

<a id="sec4"></a>
## 4. Interpretacja: iloraz szans

Współczynniki regresji logistycznej są na skali LOGITU, więc nie da się ich czytać wprost
jak w regresji liniowej. Zamiast tego przekształcamy je funkcją wykładniczą — otrzymując
**iloraz szans** (*odds ratio*):

In [ ]:
ilorazy_szans = np.exp(model.params)
print(ilorazy_szans)


Iloraz szans `> 1` oznacza wyższe szanse konwersji, `< 1` — niższe, `= 1` — brak wpływu.
Dla `grupa_kampanii[T.B]` iloraz szans wynosi w przybliżeniu `2` — czyli **szanse**
(nie prawdopodobieństwo wprost, tylko stosunek `p / (1-p)`) konwersji w grupie B są około
dwukrotnie wyższe niż w grupie A.

In [ ]:
wskaznik_konwersji = klienci.groupby("grupa_kampanii")["konwersja"].mean()
print(wskaznik_konwersji)

szanse_a = wskaznik_konwersji["A"] / (1 - wskaznik_konwersji["A"])
szanse_b = wskaznik_konwersji["B"] / (1 - wskaznik_konwersji["B"])
print(f"Szanse A: {szanse_a:.3f}, szanse B: {szanse_b:.3f}")
print(f"Stosunek szans B/A: {szanse_b / szanse_a:.3f}")


> 💡 **Iloraz szans to nie to samo, co 'ile razy wyższe prawdopodobieństwo'**
>
> To jeden z najczęstszych błędów w interpretacji: iloraz szans `2` NIE znaczy, że prawdopodobieństwo jest dwa razy wyższe (`0.118 * 2 = 0.236`, a faktyczne `p` w grupie B to `0.218` — blisko, ale nie identycznie). Iloraz szans mówi o stosunku SZANS (`p/(1-p)`), nie samych prawdopodobieństw. Różnica jest mała, gdy `p` jest niewielkie, ale rośnie znacząco przy `p` bliskim `0.5` lub większym.

<a id="sec5"></a>
## 5. Przewidywanie prawdopodobieństw

Metoda `.predict()` w regresji logistycznej zwraca od razu PRAWDOPODOBIEŃSTWA (już po
przejściu przez funkcję logistyczną), a nie surowy logit:

In [ ]:
nowi_klienci = pd.DataFrame({"grupa_kampanii": ["A", "B"]})
przewidziane_prawdopodobienstwa = model.predict(nowi_klienci)
print(przewidziane_prawdopodobienstwa)


<a id="sec6"></a>
## 6. Próg decyzyjny i macierz pomyłek

Żeby zamienić prawdopodobieństwo na konkretną decyzję (konwersja tak/nie), potrzebujemy
**progu decyzyjnego** (*threshold*) — najczęściej domyślnie `0.5`. Sprawdźmy, jak dobrze
model z progiem `0.5` klasyfikuje WSZYSTKICH naszych klientów:

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score

klienci["prawdopodobienstwo"] = model.predict(klienci)
klienci["przewidziana_klasa"] = (klienci["prawdopodobienstwo"] >= 0.5).astype(int)

macierz_pomylek = confusion_matrix(klienci["konwersja"], klienci["przewidziana_klasa"])
print("Macierz pomylek:")
print(macierz_pomylek)
print(f"Dokladnosc (accuracy): {accuracy_score(klienci['konwersja'], klienci['przewidziana_klasa']):.3f}")
print(klienci["przewidziana_klasa"].value_counts())


> ⚠️ **Paradoks dokładności (accuracy paradox)**
>
> Widzisz, co się stało? Model przewiduje klasę `0` (brak konwersji) dla ZUPEŁNIE WSZYSTKICH klientów -- bo nawet w grupie B przewidywane prawdopodobieństwo (`~0.22`) nie przekracza progu `0.5`. A mimo to dokładność wychodzi wysoka (ok. 83%)! To dlatego, że sama baza konwersji jest NISKA (ok. 17% klientów konwertuje) -- model "zgadujący zawsze nie" trafia w większość przypadków, nie ucząc się kompletnie niczego przydatnego. Przy niezbalansowanych klasach sama dokładność jest myląca -- potrzebujemy lepszych miar (sekcja 7).

<a id="sec7"></a>
## 7. Precyzja i czułość

**Precyzja** (*precision*) — spośród przewidzianych konwersji, ile faktycznie było
konwersjami? **Czułość** (*recall*) — spośród faktycznych konwersji, ile model wykrył?
Przy progu `0.5` model z sekcji 6 w ogóle nie przewiduje klasy `1`, więc obie miary
wynoszą `0`. Obniżmy próg, żeby model faktycznie zaczął coś wykrywać:

In [ ]:
from sklearn.metrics import precision_score, recall_score

for prog in [0.5, 0.2, 0.15]:
    klasa_przy_progu = (klienci["prawdopodobienstwo"] >= prog).astype(int)
    precyzja = precision_score(klienci["konwersja"], klasa_przy_progu, zero_division=0)
    czulosc = recall_score(klienci["konwersja"], klasa_przy_progu, zero_division=0)
    dokladnosc = accuracy_score(klienci["konwersja"], klasa_przy_progu)
    print(f"prog={prog}: precyzja={precyzja:.3f}, czulosc={czulosc:.3f}, dokladnosc={dokladnosc:.3f}")


> 📊 **Wybór progu to kompromis, nie jedna 'poprawna' wartość**
>
> Niższy próg wykrywa więcej rzeczywistych konwersji (wyższa czułość), kosztem większej liczby fałszywych alarmów (niższa precyzja) i niższej ogólnej dokładności. Który próg jest "lepszy" zależy całkowicie od kontekstu biznesowego -- np. gdy koszt przeoczenia konwersji jest wysoki (kampania marketingowa), a koszt fałszywego alarmu niski, warto obniżyć próg.

<a id="sec8"></a>
## 8. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- dlaczego zwykła regresja liniowa nie nadaje się do przewidywania wyniku 0/1,
- funkcję logistyczną (sigmoid) i logit jako liniową kombinację predyktorów,
- dopasowanie regresji logistycznej przez `statsmodels.formula.api.logit`,
- interpretację współczynników jako ilorazów szans (`np.exp`),
- przewidywanie prawdopodobieństw przez `.predict()`,
- próg decyzyjny, macierz pomyłek i paradoks dokładności przy niezbalansowanych klasach,
- precyzję i czułość jako lepsze miary dla klas niezbalansowanych.

> 📝 **Ćwiczenie 1: Czy wiek wpływa na konwersję?**
>
> Zbuduj model `konwersja ~ C(grupa_kampanii) + wiek`. Sprawdź wartość p współczynnika przy `wiek`. Czy wiek dodaje realną wartość predykcyjną ponad to, co już daje `grupa_kampanii`?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
model_z_wiekiem = logit("konwersja ~ C(grupa_kampanii) + wiek", data=klienci).fit(disp=0)
print(model_z_wiekiem.summary())
```

Wartosc p dla wiek wychodzi wyraznie powyzej 0.05 (bez istotnosci) -- podobnie jak we wszystkich poprzednich modulach, wiek jest w naszych danych losowany calkowicie niezaleznie od konwersji, wiec nie wnosi realnej wartosci predykcyjnej.
</details>

> 📝 **Ćwiczenie 2: Interpretacja ilorazu szans w zdaniu**
>
> Policz iloraz szans dla `grupa_kampanii[T.B]` z modelu z sekcji 3. Napisz w komentarzu jedno zdanie, poprawnie interpretujące tę liczbę (pamiętaj o pułapce z sekcji 4 -- 'iloraz szans' to nie to samo, co 'iloraz prawdopodobienstw').

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
iloraz_b = np.exp(model.params["C(grupa_kampanii)[T.B]"])
print(f"Iloraz szans B vs A: {iloraz_b:.3f}")
# Szanse (nie prawdopodobienstwo!) konwersji w grupie B sa okolo 2 razy wyzsze
# niz szanse konwersji w grupie A.
```
</details>

> 📝 **Ćwiczenie 3: Macierz pomyłek przy niższym progu**
>
> Zbuduj macierz pomyłek dla progu `0.18` (blisko ogólnego wskaźnika konwersji). Porównaj liczbę fałszywych alarmów (klienci przewidziani jako konwertujący, którzy nie konwertowali) z progiem `0.5` z sekcji 6.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
klasa_018 = (klienci["prawdopodobienstwo"] >= 0.18).astype(int)
print(confusion_matrix(klienci["konwersja"], klasa_018))
```
</details>

> 📝 **Ćwiczenie 4: Przewidywanie dla nowego klienta**
>
> Użyj modelu z sekcji 3, żeby przewidzieć prawdopodobieństwo konwersji dla nowego klienta z grupy A oraz nowego klienta z grupy B naraz, w jednym DataFrame.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
nowi = pd.DataFrame({"grupa_kampanii": ["A", "B"]})
model.predict(nowi)
```
</details>

> 🔥 **Wyzwanie: czy kontrola wydatków zmienia efekt grupy kampanii?**
>
> Zbuduj model `konwersja ~ C(grupa_kampanii) + wydatki_miesieczne`. Porównaj iloraz szans dla `grupa_kampanii[T.B]` z tym modelem względem modelu z sekcji 3 (bez wydatków). Czy dodanie wydatków jako predyktora kontrolnego istotnie zmienia wniosek o efekcie grupy kampanii -- inaczej niż w wyzwaniu z Modułu 12 dla satysfakcji?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
model_pelny = logit("konwersja ~ C(grupa_kampanii) + wydatki_miesieczne", data=klienci).fit(disp=0)
print(np.exp(model_pelny.params))
print(f"Wartosc p dla wydatkow: {model_pelny.pvalues['wydatki_miesieczne']:.4f}")
```

Iloraz szans dla grupa_kampanii[T.B] zostaje niemal identyczny (ok. 2.05 vs ok. 2.08 bez wydatkow), a wydatki_miesieczne w ogole nie sa istotne (p wyraznie powyzej 0.05). To kontrast z wyzwaniem z Modulu 12: tam kontrola wydatkow ZMIENIALA wniosek o istotnosci grupy kampanii dla satysfakcji, bo satysfakcja naprawde zalezala od wydatkow. Tutaj konwersja w danych zalezy TYLKO od grupy kampanii, wiec dodanie wydatkow niczego nie zmienia -- kontrola zmiennych pomaga tylko wtedy, gdy ta zmienna faktycznie coś wyjaśnia.
</details>

## Co dalej?

Do tej pory testowaliśmy pojedyncze hipotezy i budowaliśmy modele na już zebranych
danych. W **Module 14** zajmiemy się drugą stroną testu A/B: jak DOBRZE go
zaprojektować od samego początku (wielkość próby, moc statystyczna) i jak uniknąć
częstych pułapek praktycznych (podglądanie wyników, wielokrotne porównania).